# Evo-1 LIBERO — A100/H100 compact torchao W8A16 QVLA proof-print version

Adds explicit proof prints for target layers, torchao replacement, calibration, and eval path.


Protocol fix: original Evo-1 max steps, strict success marker, student-action calibration, protocol-separated eval folder.

In [ ]:
# 0. GPU check
import torch, os, subprocess, sys, textwrap, json, re, time
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    raise RuntimeError("No GPU. Runtime > Change runtime type > GPU")


CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
BF16 supported: True


In [ ]:
# 1. Mount Drive and define paths
from google.colab import drive
drive.mount("/content/drive")

REPO = "/content/drive/MyDrive/Evo-1"
EVO = f"{REPO}/Evo_1"
LIBERO_EVAL = f"{REPO}/LIBERO_evaluation"
CKPT_DIR = "/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO"
RESULTS = "/content/drive/MyDrive/Evo-1-results/fp16_resumable"

os.makedirs(RESULTS, exist_ok=True)
print(REPO, EVO, LIBERO_EVAL, CKPT_DIR, RESULTS, sep="\n")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Evo-1
/content/drive/MyDrive/Evo-1/Evo_1
/content/drive/MyDrive/Evo-1/LIBERO_evaluation
/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO
/content/drive/MyDrive/Evo-1-results/fp16_resumable


In [ ]:
# 2. Clone fresh official Evo-1 or restore original files
%cd /content/drive/MyDrive
if not os.path.exists(REPO):
    !git clone https://github.com/MINT-SJTU/Evo-1.git Evo-1
else:
    print("Repo already exists:", REPO)

%cd "$REPO"
!git restore Evo_1/scripts/Evo1_server.py || true
!git restore LIBERO_evaluation/libero_client_4tasks.py || true
!git status --short


/content/drive/MyDrive
Repo already exists: /content/drive/MyDrive/Evo-1
/content/drive/MyDrive/Evo-1
 M .gitignore
 M Evo_1/dataset/config.yaml
 M Evo_1/ds_config.json
 M Evo_1/model/action_head/flow_matching.py
 M Evo_1/scripts/Evo1_server.py
 M MetaWorld_evaluation/mt50_evo1_client_prompt.py
 M MetaWorld_evaluation/tasks.jsonl
 M so100_evo1/lerobot-main/benchmarks/video/capture_camera_feed.py
 M so100_evo1/lerobot-main/docs/source/contributing.md
 M so100_evo1/lerobot-main/src/lerobot/policies/act/README.md
 M so100_evo1/lerobot-main/src/lerobot/policies/diffusion/README.md
 M so100_evo1/lerobot-main/src/lerobot/policies/evo1/dataset/config.yaml
 M so100_evo1/lerobot-main/src/lerobot/policies/evo1/ds_config.json
 M so100_evo1/lerobot-main/src/lerobot/policies/evo1/model/action_head/flow_matching.py
 M so100_evo1/lerobot-main/src/lerobot/policies/evo1/scripts/Evo1_server.py
 M so100_evo1/lerobot-main/src/lerobot/policies/smolvla/README.md
 M so100_evo1/lerobot-main/src/lerobot/polici

In [ ]:
# 3. Install micromamba and create envs
MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"

if not os.path.exists(MAMBA):
    !wget -qO /tmp/micromamba.tar.bz2 https://micro.mamba.pm/api/micromamba/linux-64/latest
    !mkdir -p /content/micromamba
    !tar -xjf /tmp/micromamba.tar.bz2 -C /content/micromamba bin/micromamba

!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} create -y -n Evo1 python=3.10 pip -c conda-forge || true
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} create -y -n libero python=3.8.13 pip -c conda-forge || true
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} env list


Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                     ✔ Done (0.1 sec)
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                     ✔ Done (0.1 sec)

Resolving Environment                                                                     ✔ Done (0.2 sec)

Transaction

  Prefix: /content/micromamba-root/envs/Evo1

  Updating specs:

   - python=3.10
   - pip


  Package               Version  Build                 Channel           Size
─────────────────────────────────────────────────────────────────

In [ ]:
# 4. Install/check Evo-1 server deps + torchao; A100/H100 preserve torch, Blackwell uses CUDA 12.8 torch
# Minimal install guard:
# - Do not run requirements.txt every session if the same requirements were already installed.
# - Do not force-reinstall huggingface-hub if 0.36.2 is already present.
# - Do not upgrade/replace torch on A100/H100.
MAMBA = '/content/micromamba/bin/micromamba'
MAMBA_ROOT = '/content/micromamba-root'
EVO = '/content/drive/MyDrive/Evo-1/Evo_1'
import os, subprocess, hashlib
from pathlib import Path

env = {**os.environ, 'MAMBA_ROOT_PREFIX': MAMBA_ROOT}
REQ = Path(EVO) / 'requirements.txt'
STAMP_DIR = Path('/content/drive/MyDrive/Evo-1-results/setup_stamps')
STAMP_DIR.mkdir(parents=True, exist_ok=True)
REQ_STAMP = STAMP_DIR / 'evo1_requirements_Evo1.sha256'
HF_PIN = '0.36.2'
FORCE_EVO1_DEPS = os.environ.get('FORCE_EVO1_DEPS', '0') == '1'

def _run(cmd, **kw):
    return subprocess.run(cmd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kw)

def _show_run(cmd, check=True):
    r = _run(cmd)
    print(r.stdout)
    if check and r.returncode != 0:
        raise RuntimeError('Command failed: ' + ' '.join(map(str, cmd)))
    return r

# Keep pip tooling sane; this is small compared with requirements/flash-attn.
_show_run([MAMBA,'run','-n','Evo1','python','-m','pip','install','-U','pip','setuptools','wheel'])

# requirements.txt guard: install only if requirements hash changed or user forces it.
req_hash = hashlib.sha256(REQ.read_bytes()).hexdigest()
old_hash = REQ_STAMP.read_text().strip() if REQ_STAMP.exists() else None
# Sanity-check: if key packages are missing, reinstall even if hash matches
_probe = __import__('subprocess').run(
    [MAMBA,'run','-n','Evo1','python','-c','import transformers, torch'],
    env=env, stdout=__import__('subprocess').PIPE, stderr=__import__('subprocess').STDOUT
)
if _probe.returncode != 0:
    print('KEY_IMPORT_MISSING: forcing requirements reinstall')
    FORCE_EVO1_DEPS = True

if FORCE_EVO1_DEPS or old_hash != req_hash:
    print('EVO1_REQUIREMENTS_INSTALL: hash changed or force requested')
    print('  old:', old_hash)
    print('  new:', req_hash)
    _show_run([MAMBA,'run','-n','Evo1','python','-m','pip','install','-r',str(REQ)])
    REQ_STAMP.write_text(req_hash)
else:
    print('EVO1_REQUIREMENTS_SKIP: requirements.txt hash unchanged:', req_hash)

# huggingface-hub pin guard: do not force reinstall if already correct.
hf_probe = '''
import sys
try:
    import huggingface_hub
    v = huggingface_hub.__version__
    print("HF_VERSION", v)
    raise SystemExit(0 if v == "''' + HF_PIN + '''" else 42)
except ModuleNotFoundError:
    print("HF_MISSING")
    raise SystemExit(42)
'''
r = _run([MAMBA,'run','-n','Evo1','python','-c',hf_probe])
print(r.stdout)
if r.returncode == 42:
    print('HF_INSTALL_PIN:', HF_PIN)
    _show_run([MAMBA,'run','-n','Evo1','python','-m','pip','install','huggingface-hub=='+HF_PIN])
elif r.returncode != 0:
    raise RuntimeError('huggingface_hub version probe failed')
else:
    print('HF_PIN_OK_SKIP_INSTALL:', HF_PIN)

smi = subprocess.run(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
major = int(smi.stdout.strip().splitlines()[0].split('.')[0]) if smi.returncode == 0 else 12
print('GPU_CAP:', major)

if major >= 12:
    # Blackwell path only. This is the only branch allowed to replace torch.
    print('BLACKWELL_DETECTED: installing CUDA 12.8 torch/torchao')
    subprocess.run([MAMBA,'run','-n','Evo1','python','-m','pip','install','--pre','--force-reinstall','torch','torchvision','torchaudio','torchao','--index-url','https://download.pytorch.org/whl/nightly/cu128'], env=env, check=True)
else:
    # A100/H100 path: do NOT upgrade torchao every time.
    # First prove the existing torchao has the API we need. Only repair/install if this probe fails.
    torchao_probe = r'''
import traceback
try:
    import torchao
    from torchao.quantization import Int8WeightOnlyConfig, quantize_
    print('TORCHAO_ALREADY_OK', getattr(torchao, '__version__', 'unknown'), flush=True)
except Exception:
    print('TORCHAO_NEEDS_INSTALL_OR_REPAIR', flush=True)
    traceback.print_exc()
    raise SystemExit(42)
'''
    r = subprocess.run([MAMBA,'run','-n','Evo1','python','-c',torchao_probe], env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout)
    if r.returncode == 42:
        # Repair torchao only, without allowing pip to replace torch/torchvision.
        subprocess.run([MAMBA,'run','-n','Evo1','python','-m','pip','install','--no-cache-dir','--force-reinstall','--no-deps','torchao'], env=env, check=True)
    elif r.returncode != 0:
        raise RuntimeError('TorchAO probe failed unexpectedly; see output above.')

# Final import check: print the exact failing module if anything is broken.
check = r'''
import importlib, traceback
mods = ['torch', 'transformers', 'huggingface_hub', 'torchao']
for m in mods:
    print('IMPORT_TEST_START', m, flush=True)
    try:
        mod = importlib.import_module(m)
        print('IMPORT_OK', m, getattr(mod, '__version__', 'unknown'), flush=True)
    except Exception:
        print('IMPORT_FAIL', m, flush=True)
        traceback.print_exc()
        raise
from torchao.quantization import Int8WeightOnlyConfig, quantize_
import torch, huggingface_hub, torchao
print('torch', torch.__version__, 'cuda', torch.version.cuda, flush=True)
print('cap', torch.cuda.get_device_capability() if torch.cuda.is_available() else None, flush=True)
print('torchao', getattr(torchao, '__version__', 'unknown'), flush=True)
print('hf', huggingface_hub.__version__, flush=True)
print('TORCHAO_API_OK Int8WeightOnlyConfig quantize_', flush=True)
'''
r = subprocess.run([MAMBA,'run','-n','Evo1','python','-c',check], env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode != 0:
    raise RuntimeError('Evo1 import check failed above. Fix the failing import before QVLA2.')



KEY_IMPORT_MISSING: forcing requirements reinstall
EVO1_REQUIREMENTS_INSTALL: hash changed or force requested
  old: 9f3647334696e85c48375d3927fff61d32fad49f4e1edb61e9641125e7ad313f
  new: 9f3647334696e85c48375d3927fff61d32fad49f4e1edb61e9641125e7ad313f
  Using cached transformers-4.39.0-py3-none-any.whl.metadata (134 kB)
  Using cached timm-1.0.27-py3-none-any.whl.metadata (40 kB)
  Using cached torch-2.5.1-cp310-cp310-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached torchvision-0.20.1-cp310-cp310-manylinux1_x86_64.whl.metadata (6.1 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached einop-0.0.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached diffusers-0.38.0-py3-none-any.whl.metadata (20 kB)
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached matplotlib-3.10.9-cp310-cp310-man

In [ ]:
                                                    # 5. FlashAttention for A100/H100 — Drive-cached wheel reuse
# Goal:
# - import-test first; if FlashAttention already works, skip everything
# - if a matching wheel exists on Drive, install it and skip compile
# - only build once when no matching cached wheel exists
# - save the built wheel to Drive for future Colab sessions
# - avoids reinstalling when the cached/imported binary already works

import os, re, json, subprocess
from pathlib import Path

MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"
EVO = "/content/drive/MyDrive/Evo-1/Evo_1"
FLASH_WHEEL_ROOT = Path("/content/drive/MyDrive/Evo-1/flash_attn_wheels")
FLASH_WHEEL_ROOT.mkdir(parents=True, exist_ok=True)

env = {**os.environ, "MAMBA_ROOT_PREFIX": MAMBA_ROOT}

def run_evo(cmd, check=False):
    r = subprocess.run(
        [MAMBA, "run", "-n", "Evo1", *cmd],
        cwd=EVO,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(r.stdout)
    if check and r.returncode != 0:
        raise RuntimeError("Command failed: " + " ".join(cmd))
    return r

def flash_import_ok():
    probe = r"""
import traceback
try:
    import flash_attn, flash_attn_2_cuda
    print("FLASH_ATTN_IMPORT_OK", getattr(flash_attn, "__version__", "unknown"), flush=True)
except Exception:
    print("FLASH_ATTN_IMPORT_FAIL", flush=True)
    traceback.print_exc()
    raise SystemExit(42)
"""
    r = run_evo(["python", "-c", probe], check=False)
    return r.returncode == 0

# 1) If current env already imports FlashAttention, do not reinstall/rebuild.
if flash_import_ok():
    print("FLASH_ATTN_SKIP: already importable in Evo1 env")
else:
    # 2) Get exact ABI/GPU key from the Evo1 env.
    info_code = r"""
import json, sys, torch
major, minor = torch.cuda.get_device_capability() if torch.cuda.is_available() else (-1, -1)
print(json.dumps({
    "python": f"{sys.version_info.major}.{sys.version_info.minor}",
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "arch": f"{major}.{minor}",
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
}))
"""
    r = run_evo(["python", "-c", info_code], check=True)
    info = json.loads(r.stdout.strip().splitlines()[-1])
    print("FLASH_ATTN_BUILD_KEY_INFO:", info)

    arch = info["arch"]
    if arch not in {"8.0", "9.0", "10.0", "12.0"}:
        raise RuntimeError(f"This FlashAttention cell is intended for A100/H100 only. Detected arch={arch}, device={info['device']}")

    key_raw = f"py{info['python']}_torch{info['torch']}_cu{info['cuda']}_sm{arch}"
    key = re.sub(r"[^A-Za-z0-9_.-]+", "_", key_raw)
    wheel_dir = FLASH_WHEEL_ROOT / key
    wheel_dir.mkdir(parents=True, exist_ok=True)
    print("FLASH_ATTN_WHEEL_DIR:", wheel_dir)

    cached = sorted(wheel_dir.glob("flash_attn*.whl"))
    if cached:
        print("FLASH_ATTN_CACHED_WHEEL_FOUND:", cached[-1])
        # Import failed, so remove broken package first. Reinstalls only after a failed import probe.
        run_evo(["python", "-m", "pip", "uninstall", "-y", "flash-attn", "flash_attn"], check=False)
        run_evo(["python", "-m", "pip", "install", "--no-deps", str(cached[-1])], check=True)
        if not flash_import_ok():
            raise RuntimeError("Cached FlashAttention wheel installed but import still failed; delete the cached wheel dir and rebuild.")
        print("FLASH_ATTN_CACHED_WHEEL_INSTALL_OK")
    else:
        print("FLASH_ATTN_NO_CACHED_WHEEL: building once, then saving wheel to Drive")
        build_script = f"""
set -euxo pipefail
python -m pip uninstall -y flash-attn flash_attn || true
python -m pip install -U packaging ninja
export MAX_JOBS=4
export TORCH_CUDA_ARCH_LIST="{arch}"
python -m pip wheel --no-build-isolation --no-deps --wheel-dir "{wheel_dir}" flash-attn
wheel="$(ls -t "{wheel_dir}"/flash_attn*.whl | head -n 1)"
python -m pip install --no-deps "$wheel"
python -c "import flash_attn, flash_attn_2_cuda; print('FLASH_ATTN_IMPORT_OK_AFTER_BUILD', flash_attn.__version__)"
"""
        r = run_evo(["bash", "-lc", build_script], check=False)
        if r.returncode != 0:
            raise RuntimeError("FlashAttention wheel build/install failed. See output above.")
        print("FLASH_ATTN_WHEEL_SAVED_TO_DRIVE:", wheel_dir)


FLASH_ATTN_IMPORT_FAIL
Traceback (most recent call last):
  File "<string>", line 4, in <module>
ModuleNotFoundError: No module named 'flash_attn'

{"python": "3.10", "torch": "2.12.0.dev20260407+cu128", "cuda": "12.8", "arch": "12.0", "device": "NVIDIA RTX PRO 6000 Blackwell Server Edition"}

FLASH_ATTN_BUILD_KEY_INFO: {'python': '3.10', 'torch': '2.12.0.dev20260407+cu128', 'cuda': '12.8', 'arch': '12.0', 'device': 'NVIDIA RTX PRO 6000 Blackwell Server Edition'}
FLASH_ATTN_WHEEL_DIR: /content/drive/MyDrive/Evo-1/flash_attn_wheels/py3.10_torch2.12.0.dev20260407_cu128_cu12.8_sm12.0
FLASH_ATTN_NO_CACHED_WHEEL: building once, then saving wheel to Drive
+ python -m pip uninstall -y flash-attn flash_attn
+ python -m pip install -U packaging ninja
+ export MAX_JOBS=4
+ MAX_JOBS=4
+ export TORCH_CUDA_ARCH_LIST=12.0
+ TORCH_CUDA_ARCH_LIST=12.0
+ python -m pip wheel --no-build-isolation --no-deps --wheel-dir /content/drive/MyDrive/Evo-1/flash_attn_wheels/py3.10_torch2.12.0.dev20260407_cu128_cu1

In [ ]:
# 6. Install LIBERO env
MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"
LIBERO_EVAL = "/content/drive/MyDrive/Evo-1/LIBERO_evaluation"

!cd "{LIBERO_EVAL}" && test -d LIBERO || git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install -U "pip<25.1" "setuptools<76" wheel
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install "numpy<1.24" "protobuf<4"
!cd "{LIBERO_EVAL}/LIBERO" && MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install -r requirements.txt
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0 --extra-index-url https://download.pytorch.org/whl/cu113
!cd "{LIBERO_EVAL}/LIBERO" && MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install -e .
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install websockets==13.1 huggingface_hub imageio imageio-ffmpeg opencv-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.7 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.3.0
    Uninstalling setuptools-75.3.0:
      Successfully uninstalled setuptools-75.3.0
  Attempting uninstall: pip
    Found existing installation: pip 24.3.1
    Uninstalling pip-24.3.1:
      Successfully uninstalled pip-24.3.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 5.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 82.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.2/829.2 kB 34.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.5/734.5 kB 44.9 MB/s eta 0:00:00
  Installing build dependencies ... done

In [ ]:
# 7. Download checkpoint
from huggingface_hub import snapshot_download
import os
CKPT_DIR = "/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO"
os.makedirs(CKPT_DIR, exist_ok=True)
snapshot_download(repo_id="MINT-SJTU/Evo1_LIBERO", local_dir=CKPT_DIR, local_dir_use_symlinks=False)
!find "{CKPT_DIR}" -maxdepth 2 -type f | sort | head -50


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO/checkpoint.json
/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO/config.json
/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO/mp_rank_00_model_states.pt
/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO/norm_stats.json


In [ ]:
# 8. Patch server only: checkpoint path, port 9010, websocket no-timeout
from pathlib import Path
import re

REPO = "/content/drive/MyDrive/Evo-1"
EVO = f"{REPO}/Evo_1"
CKPT_DIR = "/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO"

server_path = Path(f"{EVO}/scripts/Evo1_server.py")
txt = server_path.read_text()

txt = re.sub(
    r"ckpt_dir\s*=\s*['\"].*?['\"]",
    f'ckpt_dir = "{CKPT_DIR}"',
    txt,
    count=1,
)

txt = txt.replace("9000", "9010")

if "ping_interval=None" not in txt:
    txt = txt.replace(
        'websockets.serve(handler, "0.0.0.0", PORT)',
        'websockets.serve(handler, "0.0.0.0", PORT, ping_interval=None, ping_timeout=None, close_timeout=30)',
    )
    txt = txt.replace(
        "websockets.serve(handler, '0.0.0.0', PORT)",
        "websockets.serve(handler, '0.0.0.0', PORT, ping_interval=None, ping_timeout=None, close_timeout=30)",
    )

server_path.write_text(txt)

!grep -n "ckpt_dir\|9010\|9000\|websockets.serve\|ping_interval" "{server_path}" | tail -40

63:def load_model_and_normalizer(ckpt_dir):
64:    config = json.load(open(os.path.join(ckpt_dir, "config.json")))
65:    stats = json.load(open(os.path.join(ckpt_dir, "norm_stats.json")))
72:    ckpt_path = os.path.join(ckpt_dir, "mp_rank_00_model_states.pt")
149:    ckpt_dir = "/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO"
150:    #Example: ckpt_dir = "/home/dell/checkpoints/Evo1/Evo1_MetaWorld/"
152:    port = 9010
155:    model, normalizer = load_model_and_normalizer(ckpt_dir)
159:        async with websockets.serve(


In [ ]:
%%bash
# 9. LIBERO config

mkdir -p ~/.libero
mkdir -p /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/datasets

cat > ~/.libero/config.yaml <<'EOF'
benchmark_root: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero
bddl_files: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/bddl_files
init_states: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/init_files
datasets: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/datasets
assets: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/assets
EOF

cat ~/.libero/config.yaml

benchmark_root: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero
bddl_files: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/bddl_files
init_states: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/init_files
datasets: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/datasets
assets: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/assets


In [ ]:
# 10. Create runtime single-episode LIBERO client copy — fixed clean version
from pathlib import Path
import re

LIBERO_EVAL = "/content/drive/MyDrive/Evo-1/LIBERO_evaluation"
src = Path(f"{LIBERO_EVAL}/libero_client_4tasks.py")
dst = Path(f"{LIBERO_EVAL}/libero_client_single_episode_runtime.py")

original = src.read_text()

prefix = """
import os

# ---- Runtime single-episode controls inserted by Colab notebook ----
SINGLE_TASK_ID = int(os.environ.get("SINGLE_TASK_ID", "0"))
SINGLE_EP_INDEX = int(os.environ.get("SINGLE_EP_INDEX", "0"))
SINGLE_SUITE = os.environ.get("SINGLE_SUITE", "libero_spatial")
SINGLE_MAX_STEPS = int(os.environ.get("SINGLE_MAX_STEPS", "25"))
SINGLE_CKPT_NAME = os.environ.get(
    "SINGLE_CKPT_NAME",
    f"Evo1_FP16_{SINGLE_SUITE}_task{SINGLE_TASK_ID}_ep{SINGLE_EP_INDEX}",
)
# -------------------------------------------------------------------
"""

txt = prefix + "\n" + original

txt = re.sub(
    r"SERVER_URL\s*=\s*['\"].*?['\"]",
    'SERVER_URL = "ws://127.0.0.1:9010"',
    txt,
    count=1,
)

txt = re.sub(r"horizon\s*=\s*\d+", "horizon = 14", txt, count=1)
txt = re.sub(r"max_steps\s*=\s*\[[^\]]+\]", "max_steps = [SINGLE_MAX_STEPS]", txt, count=1)
txt = re.sub(r"task_suites\s*=\s*\[[^\]]+\]", "task_suites = [SINGLE_SUITE]", txt, count=1)
txt = re.sub(r"num_episodes\s*=\s*\d+", "num_episodes = 1", txt, count=1)
txt = re.sub(r"ckpt_name\s*=\s*f?['\"].*?['\"]", "ckpt_name = SINGLE_CKPT_NAME", txt, count=1)

txt = txt.replace("for task_id in range(num_tasks_in_suite):", "for task_id in [SINGLE_TASK_ID]:")
txt = txt.replace("for task_id in range(min(num_tasks_in_suite, 1)):", "for task_id in [SINGLE_TASK_ID]:")
txt = txt.replace("for task_id in range(min(num_tasks_in_suite, 10)):", "for task_id in [SINGLE_TASK_ID]:")

txt = txt.replace("initial_states[episode_id]", "initial_states[SINGLE_EP_INDEX]")
txt = txt.replace("init_states[episode_id]", "init_states[SINGLE_EP_INDEX]")
txt = txt.replace("for episode_id in range(num_episodes):", "for episode_id in [0]:")

txt = txt.replace(
    "async with websockets.connect(SERVER_URL) as ws:",
    "async with websockets.connect(SERVER_URL, max_size=100_000_000, ping_interval=None, ping_timeout=None, close_timeout=30) as ws:",
)

dst.write_text(txt)

print("Wrote:", dst)
print("\nTop of generated runtime client:")
print("\n".join(dst.read_text().splitlines()[:20]))

# Remove failed marker from the previous crashed run, so Cell 15 reruns it.
RESULTS = Path("/content/drive/MyDrive/Evo-1-results/fp16_resumable")
for name in [
    "fp16_resumable_debug_libero_spatial_task0_ep0.log",
    "fp16_resumable_debug_libero_spatial_task0_ep0.done.json",
]:
    p = RESULTS / name
    if p.exists():
        p.unlink()
        print("deleted failed previous file:", p)

Wrote: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/libero_client_single_episode_runtime.py

Top of generated runtime client:

import os

# ---- Runtime single-episode controls inserted by Colab notebook ----
SINGLE_TASK_ID = int(os.environ.get("SINGLE_TASK_ID", "0"))
SINGLE_EP_INDEX = int(os.environ.get("SINGLE_EP_INDEX", "0"))
SINGLE_SUITE = os.environ.get("SINGLE_SUITE", "libero_spatial")
SINGLE_MAX_STEPS = int(os.environ.get("SINGLE_MAX_STEPS", "25"))
SINGLE_CKPT_NAME = os.environ.get(
    "SINGLE_CKPT_NAME",
    f"Evo1_FP16_{SINGLE_SUITE}_task{SINGLE_TASK_ID}_ep{SINGLE_EP_INDEX}",
)
# -------------------------------------------------------------------

import asyncio
import websockets
import numpy as np
import json
import pathlib
import os


## Compact QVLA W8A16 cells

In [ ]:
# QVLA0. Compact W8A16 config — LLM all linears + action FFN default with saved diagnostics
from pathlib import Path
import os, json, subprocess, time, py_compile
REPO=Path('/content/drive/MyDrive/Evo-1'); EVO=REPO/'Evo_1'; RESULTS=Path('/content/drive/MyDrive/Evo-1-results/w8a16_compact'); RESULTS.mkdir(parents=True,exist_ok=True)
MAMBA='/content/micromamba/bin/micromamba'; MAMBA_ROOT='/content/micromamba-root'
QUANT_SCOPE='both'             # both | llm_only | action_only | none
LLM_LINEAR_SCOPE='all'         # all | mlp_only | attn_only | none ; all = q/k/v/o + gate/up/down
ACTION_LINEAR_SCOPE='ffn_only'     # ffn_only | all | none ; default preserves old action-head FFN quant path
CALIB_REQUESTS=6               # calibration kept; increase to 32 for stronger gain estimates
FORCE_RECALIB='0'
APPLY_CONTEXT_GAIN='1'; APPLY_ATM='1'; APPLY_OHB_ATTN='1'; APPLY_OHB_FF='1'
TAG=f'torchao_w8a16_{QUANT_SCOPE}_{LLM_LINEAR_SCOPE}_{ACTION_LINEAR_SCOPE}'
CALIB_MAX_STEPS=25
SERVER_SCRIPT=EVO/'scripts'/'Evo1_server_torchao_w8a16_compact.py'; SERVER_LOG=Path(f'/content/evo1_{TAG}_server.log'); SCALES_PATH=RESULTS/f'{TAG}_calibmax{CALIB_MAX_STEPS}_scales.json'
print('TAG:',TAG); print('CALIB_MAX_STEPS:',CALIB_MAX_STEPS); print('SERVER_SCRIPT:',SERVER_SCRIPT); print('SCALES_PATH:',SCALES_PATH)

EXPECTED_TARGET_COUNT=114  # LLM all Linear (98) + action FFN Linear (16)
QVLA_CONFIG={
    'quant_scope': QUANT_SCOPE,
    'llm_linear_scope': LLM_LINEAR_SCOPE,
    'action_linear_scope': ACTION_LINEAR_SCOPE,
    'expected_target_count': EXPECTED_TARGET_COUNT,
    'calib_requests': CALIB_REQUESTS,
    'calib_max_steps': CALIB_MAX_STEPS,
    'tag': TAG,
    'scales_path': str(SCALES_PATH),
    'server_log': str(SERVER_LOG),
}
print('EXPECTED_TARGET_COUNT:', EXPECTED_TARGET_COUNT)
print('QVLA_CONFIG:', json.dumps(QVLA_CONFIG, indent=2))


TAG: torchao_w8a16_both_all_ffn_only
CALIB_MAX_STEPS: 25
SERVER_SCRIPT: /content/drive/MyDrive/Evo-1/Evo_1/scripts/Evo1_server_torchao_w8a16_compact.py
SCALES_PATH: /content/drive/MyDrive/Evo-1-results/w8a16_compact/torchao_w8a16_both_all_ffn_only_calibmax25_scales.json
EXPECTED_TARGET_COUNT: 114
QVLA_CONFIG: {
  "quant_scope": "both",
  "llm_linear_scope": "all",
  "action_linear_scope": "ffn_only",
  "expected_target_count": 114,
  "calib_requests": 6,
  "calib_max_steps": 25,
  "tag": "torchao_w8a16_both_all_ffn_only",
  "scales_path": "/content/drive/MyDrive/Evo-1-results/w8a16_compact/torchao_w8a16_both_all_ffn_only_calibmax25_scales.json",
  "server_log": "/content/evo1_torchao_w8a16_both_all_ffn_only_server.log"
}


In [ ]:
# QVLA1. Local checkpoint copy + write compact torchao server script
DRIVE_CKPT=Path('/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO'); LOCAL_CKPT=Path('/content/Evo1_LIBERO')
subprocess.run(['mkdir','-p',str(LOCAL_CKPT)],check=True)
subprocess.run(['rsync','-ah','--info=progress2',str(DRIVE_CKPT)+'/',str(LOCAL_CKPT)+'/'],check=True)
SERVER_SCRIPT.write_text('\n# Compact Evo-1 torchao W8A16 server with calibration + ATM/OHB via forward hooks.\nimport sys, os, asyncio, websockets, numpy as np, cv2, json, torch, time, gc, re, importlib\nfrom pathlib import Path\nfrom PIL import Image\nfrom torchvision import transforms\nsys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))\nfrom scripts.Evo1 import EVO1\nfrom torchao.quantization import quantize_, Int8WeightOnlyConfig\n\ndef _qtypes():\n    out=[]\n    for mod,cls in [("torchao.quantization.quant_api","Int8Tensor"),("torchao.dtypes","Int8Tensor"),("torchao.dtypes","AffineQuantizedTensor"),("torchao.dtypes.affine_quantized_tensor","AffineQuantizedTensor")]:\n        try:\n            c=getattr(importlib.import_module(mod),cls)\n            if c not in out: out.append(c)\n        except Exception: pass\n    if not out: raise RuntimeError("torchao int8 tensor type not found")\n    return tuple(out)\nQTYPES=_qtypes()\n\nclass Normalizer:\n    def __init__(self, stats):\n        def pad(x):\n            x=torch.tensor(x,dtype=torch.float32); return torch.cat([x,torch.zeros(max(0,24-x.numel()))])[:24]\n        s=stats[list(stats.keys())[0]]\n        self.smin,self.smax=pad(s["observation.state"]["min"]),pad(s["observation.state"]["max"])\n        self.amin,self.amax=pad(s["action"]["min"]),pad(s["action"]["max"])\n    def normalize_state(self,x):\n        return torch.clamp(2*(x-self.smin.to(x.device,x.dtype))/(self.smax.to(x.device,x.dtype)-self.smin.to(x.device,x.dtype)+1e-8)-1,-1,1)\n    def denormalize_action(self,a):\n        if a.ndim==1: a=a.view(1,-1)\n        return (a+1)/2*(self.amax.to(a.device,a.dtype)-self.amin.to(a.device,a.dtype)+1e-8)+self.amin.to(a.device,a.dtype)\n\ndef load_model(ckpt):\n    cfg=json.load(open(os.path.join(ckpt,"config.json"))); stats=json.load(open(os.path.join(ckpt,"norm_stats.json")))\n    cfg["finetune_vlm"]=False; cfg["finetune_action_head"]=False\n    cfg["num_inference_timesteps"]=int(os.environ.get("EVO1_NUM_INFERENCE_TIMESTEPS","32"))\n    m=EVO1(cfg).eval(); sd=torch.load(os.path.join(ckpt,"mp_rank_00_model_states.pt"),map_location="cpu")\n    m.load_state_dict(sd["module"],strict=True)\n    return m.to("cuda").to(torch.bfloat16), Normalizer(stats)\n\ndef prep(d,norm):\n    imgs=[]\n    for im in d["image"]:\n        a=cv2.cvtColor(cv2.resize(np.array(im,dtype=np.uint8),(448,448)),cv2.COLOR_BGR2RGB)\n        imgs.append(transforms.ToTensor()(Image.fromarray(a)).to("cuda"))\n    s=torch.tensor(d["state"],dtype=torch.float32,device="cuda")\n    if s.ndim==1: s=s[None]\n    if s.shape[1]<24: s=torch.cat([s,torch.zeros((1,24-s.shape[1]),device="cuda")],1)\n    return imgs,norm.normalize_state(s).float(),d["prompt"],torch.tensor(d["image_mask"],dtype=torch.int32,device="cuda"),torch.tensor([d["action_mask"]],dtype=torch.int32,device="cuda")\n\ndef action_json(a,norm): return norm.denormalize_action(a.reshape(1,-1,24)[0]).detach().cpu().numpy().tolist()\n\nLLM_ATTN=re.compile(r"^embedder\\.model\\.language_model\\.(?:model\\.layers|layers)\\.\\d+\\.self_attn\\.(?:q_proj|k_proj|v_proj|o_proj)$")\nLLM_MLP=re.compile(r"^embedder\\.model\\.language_model\\.(?:model\\.layers|layers)\\.\\d+\\.mlp\\.(?:gate_proj|up_proj|down_proj)$")\nACTION_FFN=re.compile(r"^action_head\\.transformer_blocks\\.\\d+\\.ff\\.(?:0|2)$")\nACTION_ALL=re.compile(r"^action_head\\.transformer_blocks\\.\\d+\\..*$")\n\ndef target_info(model):\n    q=os.environ.get("EVO1_QVLA_QUANT_SCOPE","both").lower(); ls=os.environ.get("EVO1_QVLA_LLM_LINEAR_SCOPE","mlp_only").lower(); ac=os.environ.get("EVO1_QVLA_ACTION_LINEAR_SCOPE","ffn_only").lower()\n    la=[]; lm=[]; af=[]; aa=[]\n    for n,m in model.named_modules():\n        if isinstance(m,torch.nn.Linear):\n            if LLM_ATTN.match(n): la.append(n)\n            if LLM_MLP.match(n): lm.append(n)\n            if ACTION_FFN.match(n): af.append(n)\n            if ACTION_ALL.match(n): aa.append(n)\n    llm={"mlp_only":lm,"all":lm+la,"attn_only":la,"none":[]}[ls]; act={"ffn_only":af,"all":aa,"none":[]}[ac]\n    tgt=[]\n    if q in ("both","llm_only"): tgt+=llm\n    if q in ("both","action_only"): tgt+=act\n    return {"quant_scope":q,"llm_scope":ls,"action_scope":ac,"llm_mlp":sorted(lm),"llm_attn":sorted(la),"action_ffn":sorted(af),"action_all":sorted(aa),"target":sorted(set(tgt))}\n\ndef apply_w8(m):\n    info=target_info(m); t=set(info["target"])\n    print("[EVO1-W8A16] QUANT_CONFIG", "quant_scope=",info["quant_scope"], "llm_scope=",info["llm_scope"], "action_scope=",info["action_scope"], flush=True)\n    print("[EVO1-W8A16] TARGET_COUNTS", "llm_mlp=",len(info["llm_mlp"]), "llm_attn=",len(info["llm_attn"]), "action_ffn=",len(info["action_ffn"]), "action_all=",len(info["action_all"]), "selected=",len(t), flush=True)\n    # Hard guards: do not silently run "both" with zero matched LLM/action modules.\n    exp_mlp=int(os.environ.get("EVO1_EXPECT_LLM_MLP","42")); exp_attn=int(os.environ.get("EVO1_EXPECT_LLM_ATTN","56")); exp_ffn=int(os.environ.get("EVO1_EXPECT_ACTION_FFN","16")); exp_action_all=int(os.environ.get("EVO1_EXPECT_ACTION_ALL","24"))\n    if info["quant_scope"] in ("both","llm_only") and info["llm_scope"]!="none":\n        if info["llm_scope"] in ("mlp_only","all") and len(info["llm_mlp"]) == 0: raise RuntimeError("LLM MLP regex matched 0 modules")\n        if info["llm_scope"] in ("attn_only","all") and len(info["llm_attn"]) == 0: raise RuntimeError("LLM attention regex matched 0 modules")\n        if info["llm_scope"]=="mlp_only" and exp_mlp>0 and len(info["llm_mlp"]) != exp_mlp: raise RuntimeError(f"LLM MLP count {len(info[\'llm_mlp\'])} != expected {exp_mlp}")\n        if info["llm_scope"]=="all" and exp_mlp>0 and exp_attn>0 and (len(info["llm_mlp"])+len(info["llm_attn"])) != (exp_mlp+exp_attn): raise RuntimeError(f"LLM all-linear count {len(info[\'llm_mlp\'])+len(info[\'llm_attn\'])} != expected {exp_mlp+exp_attn}")\n    if info["quant_scope"] in ("both","action_only") and info["action_scope"]!="none":\n        if info["action_scope"]=="ffn_only" and len(info["action_ffn"]) == 0: raise RuntimeError("Action FFN regex matched 0 modules")\n        if info["action_scope"]=="ffn_only" and exp_ffn>0 and len(info["action_ffn"]) != exp_ffn: raise RuntimeError(f"Action FFN count {len(info[\'action_ffn\'])} != expected {exp_ffn}")\n        if info["action_scope"]=="all" and len(info["action_all"]) == 0: raise RuntimeError("Action all-linear regex matched 0 modules")\n        if info["action_scope"]=="all" and exp_action_all>0 and len(info["action_all"]) != exp_action_all: raise RuntimeError(f"Action all-linear count {len(info[\'action_all\'])} != expected {exp_action_all}")\n    print("[EVO1-W8A16] TARGET_ASSERTIONS_PASS", flush=True)\n    print("[EVO1-W8A16] TARGET_NAMES_BEGIN", flush=True)\n    for n in sorted(t):\n        print("[EVO1-W8A16] TARGET_NAME", n, flush=True)\n    print("[EVO1-W8A16] TARGET_NAMES_END", flush=True)\n    if info["quant_scope"]!="none" and not t: raise RuntimeError("No quant targets selected")\n    if t: quantize_(m,Int8WeightOnlyConfig(),filter_fn=lambda mod,fqn:isinstance(mod,torch.nn.Linear) and fqn in t,device="cuda")\n    rep=sorted(n for n,x in m.named_modules() if n in t and isinstance(getattr(x,"weight",None),QTYPES))\n    if set(rep)!=t: raise RuntimeError("torchao replacement mismatch missing="+repr(sorted(t-set(rep))[:20])+" extra="+repr(sorted(set(rep)-t)[:20]))\n    print("[EVO1-W8A16] TORCHAO_W8A16_REPLACED",len(rep),"/",len(t),flush=True)\n    print("[EVO1-W8A16] REPLACED_NAMES_BEGIN", flush=True)\n    for n in rep:\n        print("[EVO1-W8A16] REPLACED_NAME", n, flush=True)\n    print("[EVO1-W8A16] REPLACED_NAMES_END", flush=True)\n    if rep:\n        qtype_counts={}\n        mods=dict(m.named_modules())\n        for rn in rep:\n            ww=getattr(mods[rn],"weight",None)\n            key=type(ww).__module__+"."+type(ww).__name__\n            qtype_counts[key]=qtype_counts.get(key,0)+1\n        first=rep[0]; mod=mods[first]; w=getattr(mod,"weight",None)\n        print("[EVO1-W8A16] W8_QTYPE_PROOF", first, type(w), "dtype=", getattr(w,"dtype",None), "device=", getattr(w,"device",None), flush=True)\n        print("[EVO1-W8A16] W8_QTYPE_COUNTS", json.dumps(qtype_counts, sort_keys=True), flush=True)\n    info["replaced"]=rep; return info\n\nclass RMS:\n    def __init__(self):\n        self.s={}; self.n={}; self.ctx={"teacher_rms":0.0,"student_rms":0.0,"cos":0.0,"n":0}\n    def add(self,k,x):\n        v=float(torch.sqrt(torch.mean(x.detach().float()**2)+1e-12).cpu()); self.s[k]=self.s.get(k,0)+v; self.n[k]=self.n.get(k,0)+1\n    def mean(self,k): return self.s.get(k,0)/max(1,self.n.get(k,0))\n    def add_context(self,teacher,student):\n        t=teacher.detach().float().reshape(-1); s=student.detach().float().reshape(-1)\n        tr=float(torch.sqrt(torch.mean(t*t)+1e-12).cpu()); sr=float(torch.sqrt(torch.mean(s*s)+1e-12).cpu())\n        cos=float(((t*s).sum()/(torch.linalg.vector_norm(t)*torch.linalg.vector_norm(s)+1e-12)).cpu())\n        self.ctx["teacher_rms"]+=tr; self.ctx["student_rms"]+=sr; self.ctx["cos"]+=cos; self.ctx["n"]+=1\n\ndef kind(name):\n    if ".self_attn.q_proj" in name or ".self_attn.k_proj" in name: return "atm"\n    if ".self_attn.v_proj" in name or ".self_attn.o_proj" in name or ".attn.out_proj" in name: return "ohb_attn"\n    if ".mlp." in name or ".ff." in name: return "ohb_ff"\n    return None\n\ndef hooks(model,role,stats=None,scales=None,target_names=None):\n    hs=[]; target_names=set(target_names or [])\n    flags={"atm":os.environ.get("EVO1_QVLA_APPLY_ATM","1")!="0","ohb_attn":os.environ.get("EVO1_QVLA_APPLY_OHB_ATTN","1")!="0","ohb_ff":os.environ.get("EVO1_QVLA_APPLY_OHB_FF","1")!="0"}; gains=(scales or {}).get("module_gains",{})\n    for name,m in model.named_modules():\n        if not isinstance(m,torch.nn.Linear): continue\n        if target_names and name not in target_names: continue\n        k=kind(name)\n        if k is None: continue\n        def make(n=name,kk=k):\n            def h(mod,inp,out):\n                if stats is not None: stats.add(role+":"+n,out)\n                if scales is not None and flags.get(kk,False):\n                    g=float(gains.get(n,1.0))\n                    if g!=1.0: return out*g\n                return None\n            return h\n        hs.append(m.register_forward_hook(make()))\n    print(f"[EVO1-W8A16] HOOKS_REGISTERED role={role} count={len(hs)} target_filtered={bool(target_names)}", flush=True)\n    return hs\n\ndef finalize(stats,info,n):\n    names=set(k.split(":",1)[1] for k in stats.s if k.startswith("teacher:")) | set(k.split(":",1)[1] for k in stats.s if k.startswith("student:"))\n    gains={}; ratios=[]; cats={"atm":[],"ohb_attn":[],"ohb_ff":[],"other":[]}\n    for x in names:\n        tr=stats.mean("teacher:"+x); sr=stats.mean("student:"+x)\n        g=max(0.25,min(4.0,tr/max(sr,1e-8))) if tr and sr else 1.0\n        r=sr/max(tr,1e-8) if tr else 1.0\n        gains[x]=g; ratios.append(r); cats.get(kind(x) or "other", cats["other"]).append(r)\n    cn=max(1,int(stats.ctx.get("n",0)))\n    ctx_teacher=stats.ctx.get("teacher_rms",0.0)/cn; ctx_student=stats.ctx.get("student_rms",0.0)/cn; ctx_cos=stats.ctx.get("cos",0.0)/cn\n    ctx_gain=max(0.25,min(4.0,ctx_teacher/max(ctx_student,1e-8))) if ctx_teacher and ctx_student else 1.0\n    def mean(xs): return float(sum(xs)/max(1,len(xs)))\n    return {"method":"torchao W8A16 static weight-only + fixed calibration + context/ATM/OHB output gains","formula_version":"compact_w8a16_qvla_v3_alllinear_diagproof","quant_scope":info.get("quant_scope"),"llm_linear_scope":info.get("llm_scope"),"action_linear_scope":info.get("action_scope"),"calibration_requests":n,"calibration_max_steps":int(os.environ.get("EVO1_QVLA_CALIB_MAX_STEPS","25")),"target_count":len(info.get("target",[])),"target_info":info,"target_names_sorted":sorted(info["target"]),"module_gains":gains,"module_ratio_before_mean":mean(ratios),"atm_ratio_before_mean":mean(cats["atm"]),"ohb_attn_ratio_before_mean":mean(cats["ohb_attn"]),"ohb_ff_ratio_before_mean":mean(cats["ohb_ff"]),"context_teacher_rms_mean":ctx_teacher,"context_student_rms_mean":ctx_student,"context_rms_ratio_before":ctx_student/max(ctx_teacher,1e-8) if ctx_teacher else 1.0,"context_cosine_mean":ctx_cos,"context_gain":ctx_gain,"apply_flags":{"context_gain":os.environ.get("EVO1_QVLA_APPLY_CONTEXT_GAIN","1"),"atm":os.environ.get("EVO1_QVLA_APPLY_ATM","1"),"ohb_attn":os.environ.get("EVO1_QVLA_APPLY_OHB_ATTN","1"),"ohb_ff":os.environ.get("EVO1_QVLA_APPLY_OHB_FF","1")},"torchao":{"backend":"Int8WeightOnlyConfig","wbits":8,"activation_quantization":"none"}}\n\nclass State:\n    def __init__(self):\n        self.ckpt=os.environ.get("EVO1_CKPT_DIR","/content/Evo1_LIBERO"); self.n=int(os.environ.get("EVO1_QVLA_CALIB_REQUESTS","6")); self.i=0; self.done=False; self.path=Path(os.environ.get("EVO1_QVLA_SCALES_PATH","/content/drive/MyDrive/Evo-1-results/w8a16_compact_scales.json")); self.force=os.environ.get("EVO1_QVLA_FORCE_RECALIB","0") in ("1","true","True"); self.stats=RMS(); self.hs=[]\n    def load(self):\n        print("[EVO1-W8A16] torch",torch.__version__,flush=True)\n        if self.path.exists() and not self.force:\n            self.scales=json.loads(self.path.read_text())\n            saved=sorted(self.scales.get("target_names_sorted",[]) or [])\n            saved_n=int(self.scales.get("calibration_requests",0) or 0)\n            saved_calib_max=int(self.scales.get("calibration_max_steps",-1) or -1)\n            expected_calib_max=int(os.environ.get("EVO1_QVLA_CALIB_MAX_STEPS","25"))\n            required_diag=["quant_scope","llm_linear_scope","action_linear_scope","calibration_max_steps","target_count","target_names_sorted","module_gains","module_ratio_before_mean","context_teacher_rms_mean","context_student_rms_mean","context_rms_ratio_before","context_cosine_mean","context_gain","atm_ratio_before_mean","ohb_attn_ratio_before_mean","ohb_ff_ratio_before_mean","apply_flags","torchao"]\n            missing_diag=[k for k in required_diag if k not in self.scales or self.scales.get(k) is None]\n            if not saved:\n                print("[EVO1-W8A16] EXISTING_SCALES_MISSING_TARGET_NAMES -> IGNORE_AND_RECALIBRATE",self.path,flush=True)\n            elif missing_diag:\n                print("[EVO1-W8A16] EXISTING_SCALES_MISSING_DIAGNOSTICS -> IGNORE_AND_RECALIBRATE",missing_diag,flush=True)\n            elif saved_n < self.n:\n                print("[EVO1-W8A16] EXISTING_SCALES_TOO_FEW_CALIB_REQUESTS -> IGNORE_AND_RECALIBRATE",saved_n,"<",self.n,flush=True)\n            elif saved_calib_max != expected_calib_max:\n                print("[EVO1-W8A16] EXISTING_SCALES_CALIB_MAX_STEPS_MISMATCH -> IGNORE_AND_RECALIBRATE",saved_calib_max,"!=",expected_calib_max,flush=True)\n            else:\n                self.student,self.norm=load_model(self.ckpt); self.info=apply_w8(self.student)\n                if saved!=sorted(self.info["target"]):\n                    print("[EVO1-W8A16] EXISTING_SCALES_TARGET_MISMATCH -> IGNORE_AND_RECALIBRATE",flush=True)\n                    del self.student; self.student=None; gc.collect(); torch.cuda.empty_cache()\n                else:\n                    self.hs+=hooks(self.student,"student",scales=self.scales,target_names=self.info["target"]); self.done=True; print("[EVO1-W8A16] CALIBRATION_SKIPPED True",flush=True); print("[EVO1-W8A16] LOADED_EXISTING_W8A16_SCALES",self.path,flush=True); print("[EVO1-W8A16] LOADED_SCALES_TARGET_COUNT",len(saved),flush=True); print("[EVO1-W8A16] LOADED_SCALES_CALIB_REQUESTS",saved_n,flush=True); print("[EVO1-W8A16] LOADED_SCALES_CALIB_MAX_STEPS",saved_calib_max,flush=True); print("[EVO1-W8A16] CONTEXT_COSINE_MEAN",self.scales.get("context_cosine_mean"),flush=True); print("[EVO1-W8A16] CONTEXT_GAIN",self.scales.get("context_gain"),flush=True); print("[EVO1-W8A16] ATM_RATIO_BEFORE_MEAN",self.scales.get("atm_ratio_before_mean"),flush=True); print("[EVO1-W8A16] OHB_ATTN_RATIO_BEFORE_MEAN",self.scales.get("ohb_attn_ratio_before_mean"),flush=True); print("[EVO1-W8A16] OHB_FF_RATIO_BEFORE_MEAN",self.scales.get("ohb_ff_ratio_before_mean"),flush=True); return\n        self.teacher,self.norm=load_model(self.ckpt); self.student,_=load_model(self.ckpt); self.info=apply_w8(self.student); self.hs+=hooks(self.teacher,"teacher",stats=self.stats,target_names=self.info["target"])+hooks(self.student,"student",stats=self.stats,target_names=self.info["target"]); print("[EVO1-W8A16] CALIBRATION_SKIPPED False",flush=True)\n    def finish(self):\n        self.scales=finalize(self.stats,self.info,self.i); self.path.parent.mkdir(parents=True,exist_ok=True); self.path.write_text(json.dumps(self.scales,indent=2))\n        for h in self.hs: h.remove()\n        self.hs=[]; del self.teacher; self.teacher=None; gc.collect(); torch.cuda.empty_cache(); self.hs+=hooks(self.student,"student",scales=self.scales,target_names=self.info["target"]); self.done=True\n        print("[EVO1-W8A16] W8A16-CALIB-DONE",flush=True); print("[EVO1-W8A16] SCALES_WRITTEN",self.path,flush=True); print("[EVO1-W8A16] CALIB_COLLECTED",self.i,"/",self.n,flush=True); print("[EVO1-W8A16] SAVED_SCALES_TARGET_COUNT",len(self.scales.get("target_names_sorted",[])),flush=True); print("[EVO1-W8A16] SAVED_MODULE_GAIN_COUNT",len(self.scales.get("module_gains",{})),flush=True)\n        print("[EVO1-W8A16] CONTEXT_COSINE_MEAN",self.scales.get("context_cosine_mean"),flush=True); print("[EVO1-W8A16] CONTEXT_RMS_RATIO_BEFORE",self.scales.get("context_rms_ratio_before"),flush=True); print("[EVO1-W8A16] CONTEXT_GAIN",self.scales.get("context_gain"),flush=True)\n        print("[EVO1-W8A16] ATM_RATIO_BEFORE_MEAN",self.scales.get("atm_ratio_before_mean"),flush=True); print("[EVO1-W8A16] OHB_ATTN_RATIO_BEFORE_MEAN",self.scales.get("ohb_attn_ratio_before_mean"),flush=True); print("[EVO1-W8A16] OHB_FF_RATIO_BEFORE_MEAN",self.scales.get("ohb_ff_ratio_before_mean"),flush=True)\n\nasync def handle(ws,st):\n    async for msg in ws:\n        d=json.loads(msg); st.i+=1; imgs,s,p,im,am=prep(d,st.norm); seed=123450+st.i\n        if not st.done:\n            with torch.no_grad(),torch.amp.autocast("cuda",dtype=torch.bfloat16):\n                fp=st.teacher.get_vl_embeddings(images=imgs,image_mask=im,prompt=p,return_cls_only=None); qq=st.student.get_vl_embeddings(images=imgs,image_mask=im,prompt=p,return_cls_only=None); st.stats.add_context(fp,qq); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); act_teacher=st.teacher.predict_action(fp,s,action_mask=am); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); act_student=st.student.predict_action(qq,s,action_mask=am); act=act_student\n            if st.i>=st.n: st.finish()\n            await ws.send(json.dumps(action_json(act,st.norm))); print("[EVO1-W8A16] CALIB",st.i,"/",st.n,"ACTION_SOURCE=student",flush=True)\n        else:\n            with torch.no_grad(),torch.amp.autocast("cuda",dtype=torch.bfloat16):\n                q=st.student.get_vl_embeddings(images=imgs,image_mask=im,prompt=p,return_cls_only=None);\n                if os.environ.get("EVO1_QVLA_APPLY_CONTEXT_GAIN","1")!="0": q=q*float(st.scales.get("context_gain",1.0));\n                act=st.student.predict_action(q,s,action_mask=am)\n            await ws.send(json.dumps(action_json(act,st.norm))); print("[EVO1-W8A16] EVAL",st.i,flush=True)\n\nif __name__=="__main__":\n    st=State(); st.load(); port=int(os.environ.get("EVO1_PORT","9010"))\n    async def main():\n        print(f"EVO1 compact torchao W8A16 server ws://0.0.0.0:{port}",flush=True)\n        async with websockets.serve(lambda ws:handle(ws,st),"0.0.0.0",port,max_size=100_000_000,ping_interval=None,ping_timeout=None,close_timeout=30): await asyncio.Future()\n    asyncio.run(main())\n')
py_compile.compile(str(SERVER_SCRIPT),doraise=True)
print('WROTE_AND_COMPILED:',SERVER_SCRIPT)


WROTE_AND_COMPILED: /content/drive/MyDrive/Evo-1/Evo_1/scripts/Evo1_server_torchao_w8a16_compact.py


In [ ]:
# QVLA2. Start compact W8A16 server with port cleanup
subprocess.run("ss -ltnp | grep ':9010' || echo 'no process currently listening on 9010'",shell=True)
subprocess.run('fuser -k 9010/tcp || true',shell=True); time.sleep(1)
env={**os.environ,'MAMBA_ROOT_PREFIX':MAMBA_ROOT,'EVO1_CKPT_DIR':'/content/Evo1_LIBERO','EVO1_PORT':'9010','EVO1_QVLA_QUANT_SCOPE':QUANT_SCOPE,'EVO1_QVLA_LLM_LINEAR_SCOPE':LLM_LINEAR_SCOPE,'EVO1_QVLA_ACTION_LINEAR_SCOPE':ACTION_LINEAR_SCOPE,'EVO1_QVLA_CALIB_REQUESTS':str(CALIB_REQUESTS),'EVO1_QVLA_CALIB_MAX_STEPS':str(CALIB_MAX_STEPS),'EVO1_QVLA_FORCE_RECALIB':FORCE_RECALIB,'EVO1_QVLA_SCALES_PATH':str(SCALES_PATH),'EVO1_QVLA_APPLY_CONTEXT_GAIN':APPLY_CONTEXT_GAIN,'EVO1_QVLA_APPLY_ATM':APPLY_ATM,'EVO1_QVLA_APPLY_OHB_ATTN':APPLY_OHB_ATTN,'EVO1_QVLA_APPLY_OHB_FF':APPLY_OHB_FF,'EVO1_EXPECT_LLM_MLP':'42','EVO1_EXPECT_LLM_ATTN':'56','EVO1_EXPECT_ACTION_FFN':'16','EVO1_EXPECT_ACTION_ALL':'24'}
proc=subprocess.Popen([MAMBA,'run','-n','Evo1','python','-u',str(SERVER_SCRIPT)],cwd=str(EVO),env=env,stdout=open(SERVER_LOG,'w'),stderr=subprocess.STDOUT)
print('server pid:',proc.pid); print('server log:',SERVER_LOG)
for _ in range(90):
    txt=SERVER_LOG.read_text(errors='ignore') if SERVER_LOG.exists() else ''
    if 'EVO1 compact torchao W8A16 server' in txt or 'CALIBRATION_SKIPPED True' in txt:
        print('SERVER_READY'); break
    if 'Traceback' in txt or 'RuntimeError' in txt:
        print(txt[-5000:]); raise RuntimeError('server failed')
    time.sleep(2)
else:
    print(txt[-5000:]); raise TimeoutError('server did not become ready')
subprocess.run("ss -ltnp | grep ':9010'",shell=True,check=False)
print('\n'.join(SERVER_LOG.read_text(errors='ignore').splitlines()[-80:]))


server pid: 45476
server log: /content/evo1_torchao_w8a16_both_all_ffn_only_server.log
SERVER_READY
[EVO1-W8A16] REPLACED_NAME embedder.model.language_model.model.layers.11.self_attn.o_proj
[EVO1-W8A16] REPLACED_NAME embedder.model.language_model.model.layers.11.self_attn.q_proj
[EVO1-W8A16] REPLACED_NAME embedder.model.language_model.model.layers.11.self_attn.v_proj
[EVO1-W8A16] REPLACED_NAME embedder.model.language_model.model.layers.12.mlp.down_proj
[EVO1-W8A16] REPLACED_NAME embedder.model.language_model.model.layers.12.mlp.gate_proj
[EVO1-W8A16] REPLACED_NAME embedder.model.language_model.model.layers.12.mlp.up_proj
[EVO1-W8A16] REPLACED_NAME embedder.model.language_model.model.layers.12.self_attn.k_proj
[EVO1-W8A16] REPLACED_NAME embedder.model.language_model.model.layers.12.self_attn.o_proj
[EVO1-W8A16] REPLACED_NAME embedder.model.language_model.model.layers.12.self_attn.q_proj
[EVO1-W8A16] REPLACED_NAME embedder.model.language_model.model.layers.12.self_attn.v_proj
[EVO1-W8A16

In [ ]:
# QVLA3. Calibration collection/proof — trust running server state, not just SCALES_PATH.exists()
LIBERO_EVAL='/content/drive/MyDrive/Evo-1/LIBERO_evaluation'
calib_log=RESULTS/f'{TAG}_calibration.log'

def server_listening():
    return subprocess.run("ss -ltnp | grep ':9010'",shell=True,stdout=subprocess.PIPE).returncode==0

def _server_text():
    return SERVER_LOG.read_text(errors='ignore') if SERVER_LOG.exists() else ''

def _calib_ready(txt):
    return ('[EVO1-W8A16] CALIBRATION_SKIPPED True' in txt) or ('[EVO1-W8A16] W8A16-CALIB-DONE' in txt)

if not server_listening():
    raise RuntimeError('Server is not listening on 9010')

stxt=_server_text()
if '[EVO1-W8A16] CALIBRATION_SKIPPED True' in stxt:
    print('CALIB_PROOF: running server loaded existing matching W8A16 scales')
elif '[EVO1-W8A16] W8A16-CALIB-DONE' in stxt:
    print('CALIB_PROOF: running server already completed calibration')
else:
    print('CALIB_REQUIRED_BY_SERVER: running fixed calibration client')
    env={**os.environ,'MAMBA_ROOT_PREFIX':MAMBA_ROOT,'SINGLE_SUITE':'libero_spatial','SINGLE_TASK_ID':'0','SINGLE_EP_INDEX':'0','SINGLE_MAX_STEPS':str(CALIB_MAX_STEPS),'SINGLE_CKPT_NAME':f'{TAG}_calib'}
    with open(calib_log,'w') as f:
        p=subprocess.Popen([MAMBA,'run','-n','libero','python','-u','libero_client_single_episode_runtime.py'],cwd=LIBERO_EVAL,env=env,stdout=f,stderr=subprocess.STDOUT,text=True)
    for _ in range(600):
        stxt=_server_text()
        if _calib_ready(stxt):
            print('CALIB_DONE_OR_LOADED')
            break
        if 'Traceback' in stxt or 'RuntimeError' in stxt:
            print(stxt[-5000:])
            raise RuntimeError('server failed during calibration')
        if p.poll() is not None and not _calib_ready(stxt):
            print(stxt[-4000:])
            raise RuntimeError('client ended before calibration completed/loaded')
        time.sleep(2)
    else:
        print(_server_text()[-5000:])
        raise TimeoutError('calibration timed out')
    try:
        p.terminate()
    except Exception:
        pass

stxt=_server_text()
if not _calib_ready(stxt):
    print(stxt[-5000:])
    raise RuntimeError('Calibration was escaped: server did not load existing scales and did not print W8A16-CALIB-DONE.')

if not SCALES_PATH.exists():
    raise RuntimeError(f'Missing scales file after calibration/load: {SCALES_PATH}')
sd=json.loads(SCALES_PATH.read_text())
tn=sd.get('target_names_sorted') or []
if not tn:
    raise RuntimeError('Bad scales: target_names_sorted missing/empty')
if int(sd.get('calibration_requests',0) or 0) < CALIB_REQUESTS:
    raise RuntimeError(f'Bad scales: calibration_requests {sd.get("calibration_requests")} < CALIB_REQUESTS {CALIB_REQUESTS}')
if int(sd.get('calibration_max_steps',-1)) != CALIB_MAX_STEPS:
    raise RuntimeError(f'Bad scales: calibration_max_steps {sd.get("calibration_max_steps")} != CALIB_MAX_STEPS {CALIB_MAX_STEPS}; stale/wrong-protocol scales loaded.')
if sd.get('torchao',{}).get('backend') != 'Int8WeightOnlyConfig':
    raise RuntimeError('Bad scales: torchao backend is not Int8WeightOnlyConfig')
required_diag=['quant_scope','llm_linear_scope','action_linear_scope','calibration_max_steps','target_count','target_names_sorted','module_gains','module_ratio_before_mean','context_teacher_rms_mean','context_student_rms_mean','context_rms_ratio_before','context_cosine_mean','context_gain','atm_ratio_before_mean','ohb_attn_ratio_before_mean','ohb_ff_ratio_before_mean','apply_flags','torchao']
missing_diag=[k for k in required_diag if k not in sd or sd.get(k) is None]
if missing_diag:
    raise RuntimeError('Bad scales: missing new diagnostics '+repr(missing_diag)+'; stale scales were loaded. Kill server, delete this SCALES_PATH, rerun QVLA2/QVLA3.')
# Exact target-count guard for this notebook's intended QuantVLA-style scope:
# LLM all Linear = 98, action-head FFN Linear = 16, total = 114.
expected_targets = EXPECTED_TARGET_COUNT if 'EXPECTED_TARGET_COUNT' in globals() else 114
if sd.get('quant_scope') != QUANT_SCOPE or sd.get('llm_linear_scope') != LLM_LINEAR_SCOPE or sd.get('action_linear_scope') != ACTION_LINEAR_SCOPE:
    raise RuntimeError(
        'Bad scales: config mismatch. '
        f"scales=({sd.get('quant_scope')},{sd.get('llm_linear_scope')},{sd.get('action_linear_scope')}) "
        f"current=({QUANT_SCOPE},{LLM_LINEAR_SCOPE},{ACTION_LINEAR_SCOPE}). Rerun QVLA0/QVLA2/QVLA3 with the intended config."
    )
if int(sd.get('target_count', -1)) != expected_targets or len(tn) != expected_targets:
    raise RuntimeError(f'Bad target count: scales target_count={sd.get("target_count")} names={len(tn)}, expected {expected_targets}')

print('CALIB_PROOF_OK:', 'loaded_existing' if '[EVO1-W8A16] CALIBRATION_SKIPPED True' in stxt else 'newly_collected')
print('CALIB_REQUESTS_PROOF:', sd.get('calibration_requests'), '/', CALIB_REQUESTS)
print('CALIB_MAX_STEPS_PROOF:', sd.get('calibration_max_steps'), '/', CALIB_MAX_STEPS)
print('W8A16_TARGET_PROOF: target_names_sorted_count=', len(tn))
print('W8A16_TARGET_NAMES_BEGIN')
for _n in tn:
    print('W8A16_TARGET_NAME', _n)
print('W8A16_TARGET_NAMES_END')
print('MODULE_GAIN_COUNT_PROOF:', len(sd.get('module_gains',{})))
print('W8A16_BACKEND_PROOF:', sd.get('torchao'))
print('APPLY_FLAGS_PROOF:', sd.get('apply_flags'))
print('CONTEXT_COSINE_MEAN_PROOF:', sd.get('context_cosine_mean'))
print('CONTEXT_RMS_RATIO_BEFORE_PROOF:', sd.get('context_rms_ratio_before'))
print('CONTEXT_GAIN_PROOF:', sd.get('context_gain'))
print('MODULE_RATIO_BEFORE_MEAN_PROOF:', sd.get('module_ratio_before_mean'))
print('ATM_RATIO_BEFORE_MEAN_PROOF:', sd.get('atm_ratio_before_mean'))
print('OHB_ATTN_RATIO_BEFORE_MEAN_PROOF:', sd.get('ohb_attn_ratio_before_mean'))
print('OHB_FF_RATIO_BEFORE_MEAN_PROOF:', sd.get('ohb_ff_ratio_before_mean'))
print('SCALES_PATH:',SCALES_PATH)

print('SERVER_PROOF_LINES_BEGIN')
for _line in stxt.splitlines():
    if any(_k in _line for _k in ['QUANT_CONFIG','TARGET_COUNTS','TARGET_ASSERTIONS_PASS','TORCHAO_W8A16_REPLACED','W8_QTYPE_PROOF','W8_QTYPE_COUNTS','HOOKS_REGISTERED','CONTEXT_COSINE_MEAN','CONTEXT_GAIN','ATM_RATIO_BEFORE_MEAN','OHB_ATTN_RATIO_BEFORE_MEAN','OHB_FF_RATIO_BEFORE_MEAN','CALIBRATION_SKIPPED','W8A16-CALIB-DONE','CALIB_COLLECTED','LOADED_SCALES_TARGET_COUNT','SAVED_SCALES_TARGET_COUNT']):
        print(_line)
print('SERVER_PROOF_LINES_END')
print('\n'.join(stxt.splitlines()[-100:]))


CALIB_REQUIRED_BY_SERVER: running fixed calibration client
CALIB_DONE_OR_LOADED
CALIB_PROOF_OK: newly_collected
CALIB_REQUESTS_PROOF: 6 / 6
CALIB_MAX_STEPS_PROOF: 25 / 25
W8A16_TARGET_PROOF: target_names_sorted_count= 114
W8A16_TARGET_NAMES_BEGIN
W8A16_TARGET_NAME action_head.transformer_blocks.0.ff.0
W8A16_TARGET_NAME action_head.transformer_blocks.0.ff.2
W8A16_TARGET_NAME action_head.transformer_blocks.1.ff.0
W8A16_TARGET_NAME action_head.transformer_blocks.1.ff.2
W8A16_TARGET_NAME action_head.transformer_blocks.2.ff.0
W8A16_TARGET_NAME action_head.transformer_blocks.2.ff.2
W8A16_TARGET_NAME action_head.transformer_blocks.3.ff.0
W8A16_TARGET_NAME action_head.transformer_blocks.3.ff.2
W8A16_TARGET_NAME action_head.transformer_blocks.4.ff.0
W8A16_TARGET_NAME action_head.transformer_blocks.4.ff.2
W8A16_TARGET_NAME action_head.transformer_blocks.5.ff.0
W8A16_TARGET_NAME action_head.transformer_blocks.5.ff.2
W8A16_TARGET_NAME action_head.transformer_blocks.6.ff.0
W8A16_TARGET_NAME action_

In [ ]:
# QVLA3c. Read-only calibration/stat proof print
# Safe: this cell does not modify server, model, scales, or eval. It only prints current saved stats.
import json, glob
from pathlib import Path

def _latest_existing(patterns):
    paths=[]
    for pat in patterns:
        paths += [Path(x) for x in glob.glob(str(pat), recursive=True) if Path(x).exists()]
    return max(paths, key=lambda p: p.stat().st_mtime) if paths else None

scales_path = Path(globals().get('SCALES_PATH', '')) if globals().get('SCALES_PATH', None) else None
server_log = Path(globals().get('SERVER_LOG', '')) if globals().get('SERVER_LOG', None) else None

if scales_path is None or not scales_path.exists():
    scales_path = _latest_existing([
        '/content/drive/MyDrive/Evo-1-results/**/*w8a16*scale*.json',
        '/content/drive/MyDrive/Evo-1-results/**/*scales*.json',
    ])
if server_log is None or not server_log.exists():
    server_log = _latest_existing([
        '/content/*w8a16*server*.log',
        '/content/evo1_torchao_w8a16*_server.log',
        '/content/evo1_*server.log',
    ])

print('======== QVLA3c CURRENT W8A16 CALIBRATION / QUANT PROOF ========')
print('SCALES_PATH:', scales_path)
print('SERVER_LOG:', server_log)
if not scales_path or not scales_path.exists():
    raise RuntimeError('No scales JSON found. Run QVLA2/QVLA3 first; calibration stats do not exist before calibration/load.')

sd = json.loads(scales_path.read_text())
stxt = server_log.read_text(errors='ignore') if server_log and server_log.exists() else ''

def show(k):
    print(f'{k}:', sd.get(k, 'MISSING'))

print('\n--- METHOD / FLAGS ---')
for k in ['quant_scope','llm_linear_scope','action_linear_scope','calibration_requests','calibration_max_steps','target_count','torchao','apply_flags']:
    show(k)

print('\n--- CONTEXT / ATM / OHB STATS ---')
# These are the old-style numbers you expected, e.g. 0.83 / 0.86 / 0.88 / 0.95.
for k in [
    'context_cosine_mean',
    'context_teacher_rms_mean',
    'context_student_rms_mean',
    'context_rms_ratio_before',
    'context_gain',
    'module_ratio_before_mean',
    'atm_ratio_before_mean',
    'ohb_attn_ratio_before_mean',
    'ohb_ff_ratio_before_mean',
]:
    show(k)

print('\n--- TARGET / W8 PROOF ---')
names = sd.get('target_names_sorted') or sd.get('target_names') or []
print('TARGET_NAMES_COUNT:', len(names))
print('EXPECTED current run LLM all + action FFN:', 114)
print('REFERENCE mlp_only + action_ffn:', 58)
print('NOTE: action_all would be 122, but this notebook intentionally keeps action_head attention FP/BF16 and quantizes action FFN only.')
print('MODULE_GAINS_COUNT:', len(sd.get('module_gains', {})))

print('\nTARGET_NAMES_BEGIN')
for i, n in enumerate(names):
    print(f'TARGET[{i:03d}]:', n)
print('TARGET_NAMES_END')

print('\n--- SERVER PROOF LINES ---')
proof_keys = [
    'QUANT_CONFIG','TARGET_COUNTS','TARGET_ASSERTIONS_PASS','TARGET_NAME',
    'TORCHAO_W8A16_REPLACED','REPLACED_NAME','W8_QTYPE_PROOF','W8_QTYPE_COUNTS',
    'HOOKS_REGISTERED','CONTEXT_COSINE_MEAN','CONTEXT_GAIN','ATM_RATIO_BEFORE_MEAN',
    'OHB_ATTN_RATIO_BEFORE_MEAN','OHB_FF_RATIO_BEFORE_MEAN','CALIBRATION_SKIPPED',
    'W8A16-CALIB-DONE','CALIB_COLLECTED','SAVED_SCALES_TARGET_COUNT','LOADED_SCALES_TARGET_COUNT',
]
for line in stxt.splitlines():
    if any(k in line for k in proof_keys):
        print(line)

print('\n--- PASS / WARNING SUMMARY ---')
print('HAS_W8A16_CALIB_DONE_OR_LOADED:', ('W8A16-CALIB-DONE' in stxt) or ('CALIBRATION_SKIPPED True' in stxt))
print('HAS_TORCHAO_REPLACED:', 'TORCHAO_W8A16_REPLACED' in stxt)
print('HAS_W8_QTYPE_PROOF:', ('W8_QTYPE_PROOF' in stxt) or ('W8_QTYPE_COUNTS' in stxt))
print('HAS_TARGET_ASSERTIONS_PASS:', 'TARGET_ASSERTIONS_PASS' in stxt)
print('HAS_CONTEXT_COSINE_IN_JSON:', 'context_cosine_mean' in sd)
print('HAS_TARGET_NAMES_SORTED_IN_JSON:', 'target_names_sorted' in sd)

if 'context_cosine_mean' not in sd:
    raise RuntimeError('context_cosine_mean missing: current scales were made by an old server or calibration did not save diagnostics. Delete stale scales or FORCE_RECALIB=True and rerun QVLA3.')
if not names:
    raise RuntimeError('target_names_sorted/target_names missing: cannot prove which linears were quantized.')
if not (('W8A16-CALIB-DONE' in stxt) or ('CALIBRATION_SKIPPED True' in stxt)):
    print('WARNING: server log does not show calibration done/loaded. Check SERVER_LOG above.')
if 'TORCHAO_W8A16_REPLACED' not in stxt:
    print('WARNING: server log does not show torchao replacement proof. Check SERVER_LOG above.')
print('======== END QVLA3c PROOF ========')


======== QVLA3c CURRENT W8A16 CALIBRATION / QUANT PROOF ========
SCALES_PATH: /content/drive/MyDrive/Evo-1-results/w8a16_compact/torchao_w8a16_both_all_ffn_only_calibmax25_scales.json
SERVER_LOG: /content/evo1_torchao_w8a16_both_all_ffn_only_server.log

--- METHOD / FLAGS ---
quant_scope: both
llm_linear_scope: all
action_linear_scope: ffn_only
calibration_requests: 6
calibration_max_steps: 25
target_count: 114
torchao: {'backend': 'Int8WeightOnlyConfig', 'wbits': 8, 'activation_quantization': 'none'}
apply_flags: {'context_gain': '1', 'atm': '1', 'ohb_attn': '1', 'ohb_ff': '1'}

--- CONTEXT / ATM / OHB STATS ---
context_cosine_mean: 0.9998507499694824
context_teacher_rms_mean: 3.729884147644043
context_student_rms_mean: 3.7253781159718833
context_rms_ratio_before: 0.9987919110905882
context_gain: 1.0012095501535376
module_ratio_before_mean: 1.0000616001348792
atm_ratio_before_mean: 0.9999841223718453
ohb_attn_ratio_before_mean: 1.0006338721413959
ohb_ff_ratio_before_mean: 0.9998227339

## Measured eval

In [ ]:
# QVLA4. Measured eval settings — 4 LIBERO suites × 10 tasks × 10 episodes, resumable
# This is the main 400-episode setting:
#   4 suites: libero_spatial, libero_object, libero_goal, libero_10
#   10 tasks per suite
#   10 measured eval episodes per task
#   total measured eval episodes = 4 * 10 * 10 = 400
#
# Calibration is excluded from this count.
# Results are saved OUTSIDE the Evo-1 repo so git restore / repo cleanup cannot delete them.

from pathlib import Path
import json, time, os

TASK_SUITES = ['libero_spatial', 'libero_object', 'libero_goal', 'libero_10']
TASK_IDS = list(range(10))
EPISODES = list(range(10))
EXPECTED_EVAL_RUNS = len(TASK_SUITES) * len(TASK_IDS) * len(EPISODES)

MAX_STEPS_BY_SUITE = {
    # Original Evo-1 LIBERO protocol: server calls, not env steps.
    # action horizon is applied inside the client; do not inflate these.
    'libero_spatial': 25,
    'libero_object': 25,
    'libero_goal': 25,
    'libero_10': 95,
}

# Keep benchmark records outside /content/drive/MyDrive/Evo-1.
# This lets you restore the repo safely and resume later.
EVAL_RESULTS = RESULTS / 'eval_4suites_10ep_protocolfix_evo1maxsteps'
EVAL_RESULTS.mkdir(parents=True, exist_ok=True)

# Safety check: EVAL_RESULTS must not be inside the repo.
# Use Path parent logic, not string startswith:
# /content/drive/MyDrive/Evo-1-results starts with the text '/content/drive/MyDrive/Evo-1'
# but it is a sibling of the repo, not inside it.
_repo_path = Path(REPO).resolve()
_eval_path = Path(EVAL_RESULTS).resolve()
if (_eval_path == _repo_path) or (_repo_path in _eval_path.parents):
    raise RuntimeError(f'EVAL_RESULTS is inside repo; unsafe for repo restore: {EVAL_RESULTS}')

manifest = {
    'tag': TAG,
    'quant_scope': QUANT_SCOPE,
    'llm_linear_scope': LLM_LINEAR_SCOPE,
    'action_linear_scope': ACTION_LINEAR_SCOPE,
    'task_suites': TASK_SUITES,
    'task_ids': TASK_IDS,
    'episodes': EPISODES,
    'expected_eval_runs': EXPECTED_EVAL_RUNS,
    'calibration_excluded_from_eval': True,
    'eval_results': str(EVAL_RESULTS),
    'max_steps_by_suite': MAX_STEPS_BY_SUITE,
    'success_rule': 'episode log contains ✅ Success only',
    'expected_target_count': 114,
    'target_scope': 'LLM all Linear (98) + action FFN Linear (16)',
    'calibration_action_source': 'student_action_sent_to_env',
    'created_time': time.time(),
}
MANIFEST_PATH = EVAL_RESULTS / f'{TAG}_benchmark_manifest.json'
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))

print('TAG:', TAG)
print('EVAL_RESULTS:', EVAL_RESULTS)
print('MANIFEST_PATH:', MANIFEST_PATH)
print('TASK_SUITES:', TASK_SUITES)
print('TASK_IDS:', TASK_IDS)
print('EPISODES:', EPISODES)
print('EXPECTED_EVAL_RUNS:', EXPECTED_EVAL_RUNS)
print('MAX_STEPS_BY_SUITE_ORIGINAL_EVO1:', MAX_STEPS_BY_SUITE)
print('SUCCESS_RULE:', '✅ Success only')
print('CALIBRATION_ACTION_SOURCE:', 'student_action_sent_to_env')
print('CALIBRATION_EXCLUDED_FROM_EVAL:', True)
print('RESULTS_OUTSIDE_REPO_PROOF:', EVAL_RESULTS, 'not under', REPO)
print('RESUME_RULE: completed episodes have *.done.json and will be skipped on rerun.')


TAG: torchao_w8a16_both_all_ffn_only
EVAL_RESULTS: /content/drive/MyDrive/Evo-1-results/w8a16_compact/eval_4suites_10ep_protocolfix_evo1maxsteps
MANIFEST_PATH: /content/drive/MyDrive/Evo-1-results/w8a16_compact/eval_4suites_10ep_protocolfix_evo1maxsteps/torchao_w8a16_both_all_ffn_only_benchmark_manifest.json
TASK_SUITES: ['libero_spatial', 'libero_object', 'libero_goal', 'libero_10']
TASK_IDS: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
EPISODES: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
EXPECTED_EVAL_RUNS: 400
MAX_STEPS_BY_SUITE_ORIGINAL_EVO1: {'libero_spatial': 25, 'libero_object': 25, 'libero_goal': 25, 'libero_10': 95}
SUCCESS_RULE: ✅ Success only
CALIBRATION_ACTION_SOURCE: student_action_sent_to_env
CALIBRATION_EXCLUDED_FROM_EVAL: True
RESULTS_OUTSIDE_REPO_PROOF: /content/drive/MyDrive/Evo-1-results/w8a16_compact/eval_4suites_10ep_protocolfix_evo1maxsteps not under /content/drive/MyDrive/Evo-1
RESUME_RULE: completed episodes have *.done.json and will be skipped on rerun.


In [ ]:
# QVLA5. Run measured eval and save per-episode summaries
# Proof before measured eval: server must be calibrated/loaded, and eval branch uses W8A16 student.
_stxt=SERVER_LOG.read_text(errors='ignore') if SERVER_LOG.exists() else ''
if not (('[EVO1-W8A16] CALIBRATION_SKIPPED True' in _stxt) or ('[EVO1-W8A16] W8A16-CALIB-DONE' in _stxt)):
    raise RuntimeError('Do not eval: W8A16 server has not loaded existing scales or completed calibration.')
_scode=SERVER_SCRIPT.read_text()
if 'act=st.student.predict_action' not in _scode or '[EVO1-W8A16] EVAL' not in _scode:
    raise RuntimeError('Server eval path proof failed: eval branch does not clearly use st.student.predict_action.')
if 'act_student=st.student.predict_action' not in _scode or 'ACTION_SOURCE=student' not in _scode:
    raise RuntimeError('Calibration path proof failed: calibration must send student action, not teacher action.')
if 'TORCHAO_W8A16_REPLACED' not in _stxt:
    raise RuntimeError('Server log lacks TORCHAO_W8A16_REPLACED proof; W8A16 replacement not confirmed.')
print('INFERENCE_PROOF_OK: calibrated/loaded W8A16 student will serve eval requests')
print('INFERENCE_PATH_PROOF: eval branch contains st.student.predict_action and prints [EVO1-W8A16] EVAL')
print('CALIBRATION_ACTION_SOURCE_PROOF: calibration branch sends st.student action to env')
print('SUCCESS_RULE_PROOF: success = (✅ Success in episode log), no total-success fallback')
print('MAX_STEPS_PROOF:', MAX_STEPS_BY_SUITE)
print('SERVER_W8A16_REPLACEMENT_PROOF:', [x for x in _stxt.splitlines() if 'TORCHAO_W8A16_REPLACED' in x][-3:])
print('SERVER_TARGET_COUNT_PROOF:', [x for x in _stxt.splitlines() if 'TARGET_COUNTS' in x][-3:])
print('SERVER_QTYPE_PROOF:', [x for x in _stxt.splitlines() if 'W8_QTYPE_PROOF' in x][-3:])
print('SERVER_REPLACED_NAMES_BEGIN')
for _x in _stxt.splitlines():
    if '[EVO1-W8A16] REPLACED_NAME' in _x:
        print(_x)
print('SERVER_REPLACED_NAMES_END')

import json, time, subprocess, os

# Local definition so QVLA5 can run after a partial notebook/kernel restart once QVLA2 started the server.
def server_listening():
    r = subprocess.run("ss -ltnp | grep ':9010'", shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return r.returncode == 0

def run_one(suite,task_id,ep):
    name=f'{TAG}_{suite}_task{task_id}_ep{ep}'; log=EVAL_RESULTS/f'{name}.log'; done=EVAL_RESULTS/f'{name}.done.json'
    if done.exists():
        try:
            prev = json.loads(done.read_text())
            result = 'SUCCESS' if bool(prev.get('success')) else 'FAIL'
            crash_prev = bool(prev.get('server_crash', prev.get('crash', False)))
            print(f'EPISODE_SKIP suite={suite} task={task_id:02d} ep={ep:02d} already_done ({result}) crash={crash_prev}', flush=True)
        except Exception as exc:
            print(f'EPISODE_SKIP suite={suite} task={task_id:02d} ep={ep:02d} already_done (unreadable_done_json: {exc})', flush=True)
        return
    if not server_listening(): raise RuntimeError('Server not listening on 9010')
    env={**os.environ,'MAMBA_ROOT_PREFIX':MAMBA_ROOT,'SINGLE_SUITE':suite,'SINGLE_TASK_ID':str(task_id),'SINGLE_EP_INDEX':str(ep),'SINGLE_MAX_STEPS':str(MAX_STEPS_BY_SUITE[suite]),'SINGLE_CKPT_NAME':name}
    with open(log,'w') as f: p=subprocess.run([MAMBA,'run','-n','libero','python','-u','libero_client_single_episode_runtime.py'],cwd=LIBERO_EVAL,env=env,stdout=f,stderr=subprocess.STDOUT,text=True)
    text=log.read_text(errors='ignore'); egl=('EGL_NOT_INITIALIZED' in text or 'EGLGLContext.__del__' in text); crash=(p.returncode!=0) or ('Traceback' in text and not (p.returncode==0 and egl)); success=('✅ Success' in text)
    rec={'suite':suite,'task_id':task_id,'episode':ep,'returncode':p.returncode,'success':success,'task_fail':(not success and not crash),'server_crash':crash,'egl_cleanup_warning':egl,'log_path':str(log),'time':time.time()}
    done.write_text(json.dumps(rec,indent=2)); print(f'EPISODE_RESULT suite={suite} task={task_id:02d} ep={ep:02d} ' + ('SUCCESS' if success else 'FAIL') + f' crash={crash}', flush=True)
    if crash or (not success):
        print('EPISODE_LOG_TAIL_BEGIN')
        print('\n'.join(text.splitlines()[-25:]))
        print('EPISODE_LOG_TAIL_END')
for s in TASK_SUITES:
    suite_done = 0
    suite_success = 0
    for t in TASK_IDS:
        for e in EPISODES:
            run_one(s,t,e)
        task_records = []
        for e in EPISODES:
            p = EVAL_RESULTS / f'{TAG}_{s}_task{t}_ep{e}.done.json'
            if p.exists():
                try:
                    task_records.append(json.loads(p.read_text()))
                except Exception as exc:
                    print(f'TASK_RESULT_READ_WARN suite={s} task={t:02d} ep={e:02d} {exc}', flush=True)
        wins = sum(1 for r in task_records if bool(r.get('success')))
        n = len(task_records)
        suite_done += n
        suite_success += wins
        print(f'TASK_RESULT suite={s} task={t:02d} success={wins}/{n} rate={(wins / n if n else 0):.3f}', flush=True)
    print(f'SUITE_PROGRESS suite={s} success={suite_success}/{suite_done} rate={(suite_success / suite_done if suite_done else 0):.3f}', flush=True)


INFERENCE_PROOF_OK: calibrated/loaded W8A16 student will serve eval requests
INFERENCE_PATH_PROOF: eval branch contains st.student.predict_action and prints [EVO1-W8A16] EVAL
CALIBRATION_ACTION_SOURCE_PROOF: calibration branch sends st.student action to env
SUCCESS_RULE_PROOF: success = (✅ Success in episode log), no total-success fallback
MAX_STEPS_PROOF: {'libero_spatial': 25, 'libero_object': 25, 'libero_goal': 25, 'libero_10': 95}
SERVER_W8A16_REPLACEMENT_PROOF: ['[EVO1-W8A16] TORCHAO_W8A16_REPLACED 114 / 114']
SERVER_TARGET_COUNT_PROOF: ['[EVO1-W8A16] TARGET_COUNTS llm_mlp= 42 llm_attn= 56 action_ffn= 16 action_all= 24 selected= 114']
SERVER_QTYPE_PROOF: ["[EVO1-W8A16] W8_QTYPE_PROOF action_head.transformer_blocks.0.ff.0 <class 'torchao.quantization.Int8Tensor'> dtype= torch.bfloat16 device= cuda:0"]
SERVER_REPLACED_NAMES_BEGIN
[EVO1-W8A16] REPLACED_NAMES_BEGIN
[EVO1-W8A16] REPLACED_NAME action_head.transformer_blocks.0.ff.0
[EVO1-W8A16] REPLACED_NAME action_head.transformer_block

In [ ]:
# QVLA6. Combine eval summaries — fixed-tag, no stale-global fallback
from pathlib import Path
import json
import pandas as pd

# This notebook is for the fixed protocol run:
#   LLM all Linear + action FFN = 114 targets
#   original Evo-1 max_steps
#   success = "✅ Success" only
DEFAULT_TAG = "torchao_w8a16_both_all_ffn_only"
DEFAULT_RESULTS = Path("/content/drive/MyDrive/Evo-1-results/w8a16_compact")
DEFAULT_EVAL_RESULTS = DEFAULT_RESULTS / "eval_4suites_10ep_protocolfix_evo1maxsteps"
EXPECTED_EVAL_RUNS = 400

# Avoid stale globals from older notebook attempts. Use the fixed tag/path for this notebook.
TAG = DEFAULT_TAG
RESULTS = DEFAULT_RESULTS
EVAL_RESULTS = DEFAULT_EVAL_RESULTS

print("COMBINE_RESULTS_DIR:", EVAL_RESULTS)
print("COMBINE_TAG_FIXED:", TAG)
print("EXPECTED_EVAL_RUNS:", EXPECTED_EVAL_RUNS)
print("STALE_GLOBAL_GUARD: QVLA6 does not reconstruct TAG from QUANT_SCOPE globals.")

manifest_files = sorted(EVAL_RESULTS.glob("*_benchmark_manifest.json"))
if manifest_files:
    print("MANIFEST_FILES_FOUND:", len(manifest_files))
    for mp in manifest_files[-3:]:
        try:
            md = json.loads(mp.read_text())
            print("MANIFEST:", mp.name, "tag=", md.get("tag"), "expected=", md.get("expected_eval_runs"))
        except Exception as exc:
            print("MANIFEST_READ_WARN:", mp, exc)

records = []
for p in sorted(EVAL_RESULTS.glob(f"{TAG}_*.done.json")):
    try:
        d = json.loads(p.read_text())
        d["_done_json"] = str(p)
        records.append(d)
    except Exception as exc:
        print("DONE_JSON_READ_WARN:", p, exc)

print("FOUND_DONE_JSON:", len(records), "/", EXPECTED_EVAL_RUNS)

if not records:
    print("No eval summaries yet for TAG:", TAG)
else:
    df = pd.DataFrame(records)

    for c in ["success", "task_fail", "server_crash", "egl_cleanup_warning"]:
        if c not in df.columns:
            df[c] = False
        df[c] = df[c].astype(bool)

    print("\n=== OVERALL ===")
    eval_runs = len(df)
    success = int(df["success"].sum())
    task_fail = int(df["task_fail"].sum())
    server_crash = int(df["server_crash"].sum())
    egl_warn = int(df["egl_cleanup_warning"].sum())
    print("completed_eval_runs:", eval_runs, "/", EXPECTED_EVAL_RUNS)
    print("success:", success)
    print("failure:", eval_runs - success)
    print("success_rate:", success / max(eval_runs, 1))
    print("task_fail:", task_fail)
    print("server_crash:", server_crash)
    print("egl_cleanup_warning:", egl_warn)

    print("\n=== BY SUITE ===")
    if "suite" in df.columns:
        by_suite = (
            df.groupby("suite")
              .agg(eval_runs=("success", "count"),
                   success=("success", "sum"),
                   task_fail=("task_fail", "sum"),
                   server_crash=("server_crash", "sum"),
                   egl_cleanup_warning=("egl_cleanup_warning", "sum"))
              .reset_index()
        )
        by_suite["failure"] = by_suite["eval_runs"] - by_suite["success"]
        by_suite["success_rate"] = by_suite["success"] / by_suite["eval_runs"].clip(lower=1)
        print(by_suite.to_string(index=False))

    print("\n=== BY TASK ===")
    if all(c in df.columns for c in ["suite", "task_id"]):
        by_task = (
            df.groupby(["suite", "task_id"])
              .agg(eval_runs=("success", "count"),
                   success=("success", "sum"),
                   task_fail=("task_fail", "sum"),
                   server_crash=("server_crash", "sum"),
                   egl_cleanup_warning=("egl_cleanup_warning", "sum"))
              .reset_index()
        )
        by_task["failure"] = by_task["eval_runs"] - by_task["success"]
        by_task["success_rate"] = by_task["success"] / by_task["eval_runs"].clip(lower=1)
        print(by_task.sort_values(["suite", "task_id"]).to_string(index=False))

    print("\n=== MISSING ===")
    expected_suites = ["libero_spatial", "libero_object", "libero_goal", "libero_10"]
    if all(c in df.columns for c in ["suite", "task_id", "episode"]):
        existing = set(zip(df["suite"], df["task_id"].astype(int), df["episode"].astype(int)))
        missing = []
        for s in expected_suites:
            for t in range(10):
                for e in range(10):
                    if (s, t, e) not in existing:
                        missing.append((s, t, e))
        print("missing:", len(missing))
        for x in missing[:120]:
            print("MISSING", x)
        if len(missing) > 120:
            print("... missing list truncated")

    # Keep the existing useful saved summaries; this does not delete or alter per-episode files.
    out_csv = EVAL_RESULTS / f"{TAG}_combined.csv"
    out_json = EVAL_RESULTS / f"{TAG}_combined.json"
    out_summary = EVAL_RESULTS / f"{TAG}_summary.json"
    df.to_csv(out_csv, index=False)
    out_json.write_text(json.dumps(records, indent=2))
    summary = {
        "tag": TAG,
        "eval_results": str(EVAL_RESULTS),
        "expected_eval_runs": EXPECTED_EVAL_RUNS,
        "completed_eval_runs": eval_runs,
        "success": success,
        "success_rate": success / max(eval_runs, 1),
        "task_fail": task_fail,
        "server_crash": server_crash,
        "egl_cleanup_warning": egl_warn,
        "calibration_excluded_from_eval": True,
        "complete_400": eval_runs == EXPECTED_EVAL_RUNS,
        "protocol": "original_evo1_max_steps_success_marker_only",
    }
    out_summary.write_text(json.dumps(summary, indent=2))
    print("\nSAVED_SUMMARIES:", out_csv, out_json, out_summary)


COMBINE_RESULTS_DIR: /content/drive/MyDrive/Evo-1-results/w8a16_compact/eval_4suites_10ep_protocolfix_evo1maxsteps
COMBINE_TAG_FIXED: torchao_w8a16_both_all_ffn_only
EXPECTED_EVAL_RUNS: 400
STALE_GLOBAL_GUARD: QVLA6 does not reconstruct TAG from QUANT_SCOPE globals.
MANIFEST_FILES_FOUND: 1
MANIFEST: torchao_w8a16_both_all_ffn_only_benchmark_manifest.json tag= torchao_w8a16_both_all_ffn_only expected= 400
FOUND_DONE_JSON: 400 / 400

=== OVERALL ===
completed_eval_runs: 400 / 400
success: 380
failure: 20
success_rate: 0.95
task_fail: 20
server_crash: 0

=== BY SUITE ===
         suite  eval_runs  success  task_fail  server_crash  egl_cleanup_warning  failure  success_rate
     libero_10        100       96          4             0                  100        4          0.96
   libero_goal        100       94          6             0                  100        6          0.94
 libero_object        100       99          1             0                  100        1          0.99
libero_sp